# 📊 คลังกราฟ EDA ฉบับแยกรูปเดี่ยว 100% (Standalone Figures for Slides)
## Thai Import & Export Stocks Predictive Pipeline (DSDE Project)
---
โน้ตบุ๊กนี้ออกแบบมาเพื่อ **แก้ปัญหาการแคปรูปยาก** โดย:
1. **แยก 1 กราฟ = 1 รูปเดี่ยว (ไม่มี Subplots รวมกัน)** สามารถคลิกขวา Copy ภาพ หรือลากไฟล์จาก `outputs/figures/` ไปวางบนสไลด์ได้ทันที
2. **แบ่งตาม 3 กลุ่มหลักชัดเจน:**
   * 🏢 **กลุ่มที่ 1: หุ้นกลุ่มนำเข้า (Import Stocks)**
   * 🚢 **กลุ่มที่ 2: หุ้นกลุ่มส่งออก (Export Stocks)**
   * 🌐 **กลุ่มที่ 3: ปัจจัยมหภาคและดัชนีโลก (17 Macro Factors)**
3. **กราฟคลีน สวยงาม ไม่มีกล่องข้อความรกในรูป** รองรับฟอนต์ไทย และเซฟความละเอียดสูง 300 DPI


In [ ]:
# ==============================================================================
# 0. Setup สภาพแวดล้อม และ โหลดข้อมูล (แก้ Duplicate Dates เรียบร้อย)
# ==============================================================================
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from matplotlib.patches import Patch

# ฟอนต์ภาษาไทย
def get_thai_font():
    candidates = [
        './Sarabun-Regular.ttf',
        '../Sarabun-Regular.ttf',
        'notebooks/Sarabun-Regular.ttf',
        '/System/Library/Fonts/Supplemental/Thonburi.ttc',
        '/System/Library/Fonts/Thonburi.ttc',
        '/Library/Fonts/Arial Unicode.ttf',
        '/usr/share/fonts/truetype/sarabun/Sarabun-Regular.ttf'
    ]
    for p in candidates:
        if Path(p).exists():
            return fm.FontProperties(fname=str(Path(p).resolve()))
    return fm.FontProperties()

thai_font = get_thai_font()
thai_font_bold = thai_font.copy()
thai_font_bold.set_weight('bold')

plt.rcParams['font.family'] = thai_font.get_name() if thai_font.get_name() else 'sans-serif'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

# โฟลเดอร์เซฟภาพ
output_fig_dir = Path("outputs/figures")
if not output_fig_dir.exists() and Path("../outputs/figures").exists():
    output_fig_dir = Path("../outputs/figures")
output_fig_dir.mkdir(parents=True, exist_ok=True)

# โหลดข้อมูล
intermediate_candidates = [
    Path('data/02_intermediate'),
    Path('../data/02_intermediate'),
    Path('./data/02_intermediate')
]
inter_dir = None
for c in intermediate_candidates:
    if c.exists():
        inter_dir = c.resolve()
        break

if not inter_dir:
    raise FileNotFoundError("❌ ไม่พบโฟลเดอร์ data/02_intermediate")

import_names = ['PTTGC', 'TOP', 'IRPC', 'IVL', 'COM7', 'BGRIM', 'GPSC', 'EGCO', 'RATCH', 'BCP']
export_names = ['DELTA', 'HANA', 'KCE', 'CPF', 'TU', 'AAI', 'STA', 'BDMS', 'BH', 'ITC']
macro_names = [
    'USD_THB', 'CNY_THB', 'EUR_THB', 'JPY_THB', 'DXY',
    'US_10Y', 'US_30Y', 'TH_10Y', 'TH_20Y', 'TH_RATE',
    'WTI', 'Brent', 'SET50', 'SP500', 'SOX', 'VIX', 'BDI'
]

price_dfs = {}
return_dfs = {}

for name in import_names + export_names + macro_names:
    p = inter_dir / f"df_{name}.csv"
    if not p.exists() and name == 'Brent':
        p = inter_dir / "df_เบรนท์.csv"
    if p.exists():
        df_tmp = pd.read_csv(p)
        df_tmp['Date'] = pd.to_datetime(df_tmp['Date'])
        df_tmp = df_tmp.sort_values('Date').dropna(subset=['Date'])
        # ป้องกัน Duplicate Dates
        df_tmp = df_tmp.drop_duplicates(subset=['Date'], keep='first')
        
        if 'Close' in df_tmp.columns:
            if df_tmp['Close'].dtype == 'object':
                df_tmp['Close'] = df_tmp['Close'].astype(str).str.replace(',', '').astype(float)
        
        if 'Pct_Change' in df_tmp.columns:
            if df_tmp['Pct_Change'].dtype == 'object':
                df_tmp['Pct_Val'] = df_tmp['Pct_Change'].astype(str).str.rstrip('%').str.replace(',', '').astype(float) / 100.0
            else:
                df_tmp['Pct_Val'] = pd.to_numeric(df_tmp['Pct_Change'], errors='coerce')
        else:
            df_tmp['Pct_Val'] = df_tmp['Close'].pct_change()
            
        df_tmp = df_tmp.dropna(subset=['Close', 'Pct_Val'])
        price_dfs[name] = df_tmp.set_index('Date')['Close']
        return_dfs[name] = df_tmp.set_index('Date')['Pct_Val']

df_prices = pd.DataFrame(price_dfs)
df_returns = pd.DataFrame(return_dfs)

print(f"✓ โหลดข้อมูลเสร็จสมบูรณ์: {df_prices.shape[1]} สินทรัพย์ (วันที่: {len(df_prices):,} วัน)")


---
# 🏢 กลุ่มที่ 1: หุ้นกลุ่มนำเข้า (Import Stocks - 10 ตัว)
ประกอบด้วย: `PTTGC`, `TOP`, `IRPC`, `IVL`, `COM7`, `BGRIM`, `GPSC`, `EGCO`, `RATCH`, `BCP`


In [ ]:
# --- 1.1 แนวโน้มราคาตามเวลา 10 ปี (Base 100) — หุ้นนำเข้า ---
fig, ax = plt.subplots(figsize=(10.5, 5.5))
imp_prices = df_prices[import_names].dropna(how='all')
imp_norm = imp_prices.apply(lambda col: (col / col.dropna().iloc[0]) * 100)

palette = sns.color_palette("tab10", len(import_names))
for i, col in enumerate(import_names):
    if col in imp_norm:
        ax.plot(imp_norm.index, imp_norm[col], label=col, lw=1.8, color=palette[i], alpha=0.85)

ax.axhline(100, color='black', linestyle='--', lw=1, alpha=0.7)
ax.set_title('แนวโน้มราคาตามเวลา: หุ้นกลุ่มนำเข้า 10 ตัว (Base = 100)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_ylabel('ดัชนีราคาเปรียบเทียบ (Base 100)', fontproperties=thai_font, fontsize=11)
ax.set_xlabel('ปี (Year)', fontproperties=thai_font, fontsize=11)
ax.legend(loc='upper left', prop=thai_font, ncol=5, framealpha=0.9, fontsize=9.5)
plt.tight_layout()

plt.savefig(output_fig_dir / "import_01_price_trend.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: import_01_price_trend.png")
plt.show()


In [ ]:
# --- 1.2 ผลตอบแทนสะสมตลอดช่วงเวลา (Total Return %) — หุ้นนำเข้า ---
fig, ax = plt.subplots(figsize=(9, 5.5))
total_ret_imp = pd.Series({
    s: ((imp_prices[s].dropna().iloc[-1] - imp_prices[s].dropna().iloc[0]) / imp_prices[s].dropna().iloc[0]) * 100
    for s in import_names if s in imp_prices
}).sort_values(ascending=True)

colors_ret = ['#2e7d32' if x >= 0 else '#c62828' for x in total_ret_imp.values]
bars = ax.barh(total_ret_imp.index, total_ret_imp.values, color=colors_ret, edgecolor='black', lw=0.6, height=0.65)
ax.axvline(0, color='black', lw=1)

for b in bars:
    w = b.get_width()
    ax.text(w + (3 if w >= 0 else -14), b.get_y() + b.get_height()/2, f"{w:+.1f}%", 
            va='center', fontsize=10, fontweight='bold', color='#1b5e20' if w >= 0 else '#b71c1c')

ax.set_title('ผลตอบแทนสะสมตลอดช่วงเวลา: หุ้นกลุ่มนำเข้า (Total Return %)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('ผลตอบแทนสะสม (%)', fontproperties=thai_font, fontsize=11)
plt.tight_layout()

plt.savefig(output_fig_dir / "import_02_total_return.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: import_02_total_return.png")
plt.show()


In [ ]:
# --- 1.3 สัดส่วนวันปิดบวก vs ปิดลบ (Up / Down Days Ratio %) — หุ้นนำเข้า ---
fig, ax = plt.subplots(figsize=(9.5, 5.5))
up_down_data = []
for s in total_ret_imp.index:
    s_ret = df_returns[s].dropna()
    up = (s_ret > 0).mean() * 100
    down = (s_ret < 0).mean() * 100
    flat = (s_ret == 0).mean() * 100
    up_down_data.append({'Stock': s, 'Up': up, 'Down': down, 'Flat': flat})

df_ud = pd.DataFrame(up_down_data).set_index('Stock')
ax.barh(df_ud.index, df_ud['Up'], color='#2e7d32', edgecolor='black', lw=0.5, label='วันปิดบวก (วันเขียว %)', height=0.65)
ax.barh(df_ud.index, df_ud['Down'], left=df_ud['Up'], color='#c62828', edgecolor='black', lw=0.5, label='วันปิดลบ (วันแดง %)', height=0.65)
ax.barh(df_ud.index, df_ud['Flat'], left=df_ud['Up'] + df_ud['Down'], color='#9e9e9e', edgecolor='black', lw=0.5, label='วันเสมอตัว (%)', height=0.65)
ax.axvline(50, color='yellow', linestyle='--', lw=1.2, label='เส้นกึ่งกลาง (50%)')

ax.set_title('สัดส่วนวันปิดบวก vs วันปิดลบ: หุ้นกลุ่มนำเข้า (%)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('สัดส่วนของจำนวนวัน (%)', fontproperties=thai_font, fontsize=11)
ax.set_xlim(0, 100)
ax.legend(loc='lower left', prop=thai_font, fontsize=10, framealpha=0.95)
plt.tight_layout()

plt.savefig(output_fig_dir / "import_03_up_down_ratio.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: import_03_up_down_ratio.png")
plt.show()


In [ ]:
# --- 1.4 วันที่ขึ้นแรงสุด vs ลงลึกสุดใน 1 วัน (%) — หุ้นนำเข้า ---
fig, ax = plt.subplots(figsize=(10, 5.8))
extreme_data = []
for s in import_names:
    s_ret = df_returns[s].dropna() * 100.0
    extreme_data.append({'Stock': s, 'Max_Gain': s_ret.max(), 'Max_Loss': s_ret.min()})

df_ext = pd.DataFrame(extreme_data).set_index('Stock').sort_values('Max_Gain', ascending=True)
y_pos = np.arange(len(df_ext))

ax.barh(y_pos, df_ext['Max_Gain'], color='#388e3c', edgecolor='black', lw=0.5, height=0.65, label='วันที่ขึ้นแรงที่สุด (Best Day %)')
ax.barh(y_pos, df_ext['Max_Loss'], color='#d32f2f', edgecolor='black', lw=0.5, height=0.65, label='วันที่ลงลึกที่สุด (Worst Day %)')

for i, (gain, loss) in enumerate(zip(df_ext['Max_Gain'], df_ext['Max_Loss'])):
    ax.text(gain + 0.6, i, f"+{gain:.1f}%", va='center', fontsize=9.5, fontweight='bold', color='#1b5e20')
    ax.text(loss - 0.6, i, f"{loss:.1f}%", va='center', ha='right', fontsize=9.5, fontweight='bold', color='#b71c1c')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_ext.index, fontproperties=thai_font_bold, fontsize=11)
ax.axvline(0, color='black', lw=1)
ax.set_title('วันที่ผันผวนสุดขีด: วันที่ขึ้นแรงสุด vs ลงลึกสุดใน 1 วัน (หุ้นนำเข้า)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('การเปลี่ยนแปลงในวันเดียว (%)', fontproperties=thai_font, fontsize=11)
ax.legend(loc='lower left', prop=thai_font, fontsize=10)
ax.set_xlim(df_ext['Max_Loss'].min() - 6, df_ext['Max_Gain'].max() + 6)
plt.tight_layout()

plt.savefig(output_fig_dir / "import_04_extreme_days.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: import_04_extreme_days.png")
plt.show()


In [ ]:
# --- 1.5 สหสัมพันธ์ภายในกลุ่มนำเข้า (Intra-group Correlation) ---
fig, ax = plt.subplots(figsize=(8.5, 7))
corr_imp = df_returns[import_names].corr()
sns.heatmap(corr_imp, annot=True, fmt='.2f', cmap='Blues', ax=ax, cbar=True,
            annot_kws={'fontsize': 10, 'fontweight': 'bold'}, vmin=0, vmax=0.7)
ax.set_title('สหสัมพันธ์ภายในกลุ่มนำเข้า (Intra-Group Correlation Matrix)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), fontproperties=thai_font_bold, rotation=45)
ax.set_yticklabels(ax.get_yticklabels(), fontproperties=thai_font_bold, rotation=0)
plt.tight_layout()

plt.savefig(output_fig_dir / "import_05_correlation_matrix.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: import_05_correlation_matrix.png")
plt.show()


---
# 🚢 กลุ่มที่ 2: หุ้นกลุ่มส่งออก (Export Stocks - 10 ตัว)
ประกอบด้วย: `DELTA`, `HANA`, `KCE`, `CPF`, `TU`, `AAI`, `STA`, `BDMS`, `BH`, `ITC`


In [ ]:
# --- 2.1 แนวโน้มราคาตามเวลา 10 ปี (Base 100) — หุ้นส่งออก ---
fig, ax = plt.subplots(figsize=(10.5, 5.5))
exp_prices = df_prices[export_names].dropna(how='all')
exp_norm = exp_prices.apply(lambda col: (col / col.dropna().iloc[0]) * 100)

palette = sns.color_palette("tab10", len(export_names))
for i, col in enumerate(export_names):
    if col in exp_norm:
        ax.plot(exp_norm.index, exp_norm[col], label=col, lw=1.8, color=palette[i], alpha=0.85)

ax.axhline(100, color='black', linestyle='--', lw=1, alpha=0.7)
ax.set_title('แนวโน้มราคาตามเวลา: หุ้นกลุ่มส่งออก 10 ตัว (Base = 100)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_ylabel('ดัชนีราคาเปรียบเทียบ (Base 100)', fontproperties=thai_font, fontsize=11)
ax.set_xlabel('ปี (Year)', fontproperties=thai_font, fontsize=11)
ax.legend(loc='upper left', prop=thai_font, ncol=5, framealpha=0.9, fontsize=9.5)
plt.tight_layout()

plt.savefig(output_fig_dir / "export_01_price_trend.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: export_01_price_trend.png")
plt.show()


In [ ]:
# --- 2.2 ผลตอบแทนสะสมตลอดช่วงเวลา (Total Return %) — หุ้นส่งออก ---
fig, ax = plt.subplots(figsize=(9, 5.5))
total_ret_exp = pd.Series({
    s: ((exp_prices[s].dropna().iloc[-1] - exp_prices[s].dropna().iloc[0]) / exp_prices[s].dropna().iloc[0]) * 100
    for s in export_names if s in exp_prices
}).sort_values(ascending=True)

colors_ret = ['#2e7d32' if x >= 0 else '#c62828' for x in total_ret_exp.values]
bars = ax.barh(total_ret_exp.index, total_ret_exp.values, color=colors_ret, edgecolor='black', lw=0.6, height=0.65)
ax.axvline(0, color='black', lw=1)

for b in bars:
    w = b.get_width()
    ax.text(w + (3 if w >= 0 else -12), b.get_y() + b.get_height()/2, f"{w:+.1f}%", 
            va='center', fontsize=10, fontweight='bold', color='#1b5e20' if w >= 0 else '#b71c1c')

ax.set_title('ผลตอบแทนสะสมตลอดช่วงเวลา: หุ้นกลุ่มส่งออก (Total Return %)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('ผลตอบแทนสะสม (%)', fontproperties=thai_font, fontsize=11)
plt.tight_layout()

plt.savefig(output_fig_dir / "export_02_total_return.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: export_02_total_return.png")
plt.show()


In [ ]:
# --- 2.3 สัดส่วนวันปิดบวก vs ปิดลบ (Up / Down Days Ratio %) — หุ้นส่งออก ---
fig, ax = plt.subplots(figsize=(9.5, 5.5))
up_down_data_exp = []
for s in total_ret_exp.index:
    s_ret = df_returns[s].dropna()
    up = (s_ret > 0).mean() * 100
    down = (s_ret < 0).mean() * 100
    flat = (s_ret == 0).mean() * 100
    up_down_data_exp.append({'Stock': s, 'Up': up, 'Down': down, 'Flat': flat})

df_ud_exp = pd.DataFrame(up_down_data_exp).set_index('Stock')
ax.barh(df_ud_exp.index, df_ud_exp['Up'], color='#2e7d32', edgecolor='black', lw=0.5, label='วันปิดบวก (วันเขียว %)', height=0.65)
ax.barh(df_ud_exp.index, df_ud_exp['Down'], left=df_ud_exp['Up'], color='#c62828', edgecolor='black', lw=0.5, label='วันปิดลบ (วันแดง %)', height=0.65)
ax.barh(df_ud_exp.index, df_ud_exp['Flat'], left=df_ud_exp['Up'] + df_ud_exp['Down'], color='#9e9e9e', edgecolor='black', lw=0.5, label='วันเสมอตัว (%)', height=0.65)
ax.axvline(50, color='yellow', linestyle='--', lw=1.2, label='เส้นกึ่งกลาง (50%)')

ax.set_title('สัดส่วนวันปิดบวก vs วันปิดลบ: หุ้นกลุ่มส่งออก (%)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('สัดส่วนของจำนวนวัน (%)', fontproperties=thai_font, fontsize=11)
ax.set_xlim(0, 100)
ax.legend(loc='lower left', prop=thai_font, fontsize=10, framealpha=0.95)
plt.tight_layout()

plt.savefig(output_fig_dir / "export_03_up_down_ratio.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: export_03_up_down_ratio.png")
plt.show()


In [ ]:
# --- 2.4 วันที่ขึ้นแรงสุด vs ลงลึกสุดใน 1 วัน (%) — หุ้นส่งออก ---
fig, ax = plt.subplots(figsize=(10, 5.8))
extreme_data_exp = []
for s in export_names:
    s_ret = df_returns[s].dropna() * 100.0
    extreme_data_exp.append({'Stock': s, 'Max_Gain': s_ret.max(), 'Max_Loss': s_ret.min()})

df_ext_exp = pd.DataFrame(extreme_data_exp).set_index('Stock').sort_values('Max_Gain', ascending=True)
y_pos = np.arange(len(df_ext_exp))

ax.barh(y_pos, df_ext_exp['Max_Gain'], color='#388e3c', edgecolor='black', lw=0.5, height=0.65, label='วันที่ขึ้นแรงที่สุด (Best Day %)')
ax.barh(y_pos, df_ext_exp['Max_Loss'], color='#d32f2f', edgecolor='black', lw=0.5, height=0.65, label='วันที่ลงลึกที่สุด (Worst Day %)')

for i, (gain, loss) in enumerate(zip(df_ext_exp['Max_Gain'], df_ext_exp['Max_Loss'])):
    ax.text(gain + 0.6, i, f"+{gain:.1f}%", va='center', fontsize=9.5, fontweight='bold', color='#1b5e20')
    ax.text(loss - 0.6, i, f"{loss:.1f}%", va='center', ha='right', fontsize=9.5, fontweight='bold', color='#b71c1c')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_ext_exp.index, fontproperties=thai_font_bold, fontsize=11)
ax.axvline(0, color='black', lw=1)
ax.set_title('วันที่ผันผวนสุดขีด: วันที่ขึ้นแรงสุด vs ลงลึกสุดใน 1 วัน (หุ้นส่งออก)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('การเปลี่ยนแปลงในวันเดียว (%)', fontproperties=thai_font, fontsize=11)
ax.legend(loc='lower left', prop=thai_font, fontsize=10)
ax.set_xlim(-95, 35)
plt.tight_layout()

plt.savefig(output_fig_dir / "export_04_extreme_days.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: export_04_extreme_days.png")
plt.show()


In [ ]:
# --- 2.5 สหสัมพันธ์ภายในกลุ่มส่งออก (Intra-group Correlation) ---
fig, ax = plt.subplots(figsize=(8.5, 7))
corr_exp = df_returns[export_names].corr()
sns.heatmap(corr_exp, annot=True, fmt='.2f', cmap='Greens', ax=ax, cbar=True,
            annot_kws={'fontsize': 10, 'fontweight': 'bold'}, vmin=0, vmax=0.7)
ax.set_title('สหสัมพันธ์ภายในกลุ่มส่งออก (Intra-Group Correlation Matrix)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), fontproperties=thai_font_bold, rotation=45)
ax.set_yticklabels(ax.get_yticklabels(), fontproperties=thai_font_bold, rotation=0)
plt.tight_layout()

plt.savefig(output_fig_dir / "export_05_correlation_matrix.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: export_05_correlation_matrix.png")
plt.show()


---
# 🌐 กลุ่มที่ 3: ปัจจัยมหภาคและดัชนีตลาดโลก (17 Macro Factors)
ครอบคลุม: FX, ดอกเบี้ย, พันธบัตร, น้ำมันดิบ, SET50, SOX, SP500, VIX


In [ ]:
# --- 3.1 แนวโน้มดัชนีมหภาคสำคัญ (Base 100) ---
fig, ax = plt.subplots(figsize=(10.5, 5.5))
key_macros = ['SET50', 'SOX', 'SP500', 'WTI', 'USD_THB', 'DXY', 'VIX']
macro_prices = df_prices[key_macros].dropna(how='all')
macro_norm = macro_prices.apply(lambda col: (col / col.dropna().iloc[0]) * 100)

palette = sns.color_palette("Set2", len(key_macros))
for i, col in enumerate(key_macros):
    if col in macro_norm:
        ax.plot(macro_norm.index, macro_norm[col], label=col, lw=2.0, color=palette[i], alpha=0.9)

ax.axhline(100, color='black', linestyle='--', lw=1, alpha=0.7)
ax.set_title('แนวโน้มดัชนีมหภาคสำคัญตามเวลา (Base = 100)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_ylabel('ดัชนีเปรียบเทียบ (Base 100)', fontproperties=thai_font, fontsize=11)
ax.set_xlabel('ปี (Year)', fontproperties=thai_font, fontsize=11)
ax.legend(loc='upper left', prop=thai_font, ncol=4, framealpha=0.9, fontsize=9.5)
plt.tight_layout()

plt.savefig(output_fig_dir / "macro_01_price_trend.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: macro_01_price_trend.png")
plt.show()


In [ ]:
# --- 3.2 การเปลี่ยนแปลงสะสมของ 17 ปัจจัยมหภาค (%) ---
fig, ax = plt.subplots(figsize=(9, 7))
total_ret_macro = pd.Series({
    m: ((df_prices[m].dropna().iloc[-1] - df_prices[m].dropna().iloc[0]) / df_prices[m].dropna().iloc[0]) * 100
    for m in macro_names if m in df_prices and len(df_prices[m].dropna()) > 1
}).sort_values(ascending=True)

colors_ret = ['#2e7d32' if x >= 0 else '#c62828' for x in total_ret_macro.values]
bars = ax.barh(total_ret_macro.index, total_ret_macro.values, color=colors_ret, edgecolor='black', lw=0.5, height=0.68)
ax.axvline(0, color='black', lw=1)

for b in bars:
    w = b.get_width()
    ax.text(w + (4 if w >= 0 else -18), b.get_y() + b.get_height()/2, f"{w:+.1f}%", 
            va='center', fontsize=9.5, fontweight='bold', color='#1b5e20' if w >= 0 else '#b71c1c')

ax.set_title('การเปลี่ยนแปลงสะสมตลอดช่วงเวลา: 17 ปัจจัยมหภาค (%)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('การเปลี่ยนแปลงสะสม (%)', fontproperties=thai_font, fontsize=11)
plt.tight_layout()

plt.savefig(output_fig_dir / "macro_02_total_return.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: macro_02_total_return.png")
plt.show()


In [ ]:
# --- 3.3 สัดส่วนวันขึ้น vs วันลง ของ 17 ปัจจัยมหภาค (%) ---
fig, ax = plt.subplots(figsize=(9.5, 7))
up_down_data_macro = []
for m in total_ret_macro.index:
    m_ret = df_returns[m].dropna()
    up = (m_ret > 0).mean() * 100
    down = (m_ret < 0).mean() * 100
    flat = (m_ret == 0).mean() * 100
    up_down_data_macro.append({'Macro': m, 'Up': up, 'Down': down, 'Flat': flat})

df_ud_macro = pd.DataFrame(up_down_data_macro).set_index('Macro')
ax.barh(df_ud_macro.index, df_ud_macro['Up'], color='#2e7d32', edgecolor='black', lw=0.5, label='วันปิดบวก (% ขึ้น)', height=0.68)
ax.barh(df_ud_macro.index, df_ud_macro['Down'], left=df_ud_macro['Up'], color='#c62828', edgecolor='black', lw=0.5, label='วันปิดลบ (% ลง)', height=0.68)
ax.barh(df_ud_macro.index, df_ud_macro['Flat'], left=df_ud_macro['Up'] + df_ud_macro['Down'], color='#9e9e9e', edgecolor='black', lw=0.5, label='วันไม่เปลี่ยนแปลง (%)', height=0.68)
ax.axvline(50, color='yellow', linestyle='--', lw=1.2, label='เส้นกึ่งกลาง (50%)')

ax.set_title('สัดส่วนวันขึ้น vs วันลง: 17 ปัจจัยมหภาค (%)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('สัดส่วนของจำนวนวัน (%)', fontproperties=thai_font, fontsize=11)
ax.set_xlim(0, 100)
ax.legend(loc='lower left', prop=thai_font, fontsize=9.5, framealpha=0.95)
plt.tight_layout()

plt.savefig(output_fig_dir / "macro_03_up_down_ratio.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: macro_03_up_down_ratio.png")
plt.show()


In [ ]:
# --- 3.4 วันที่ผันผวนสุดขีดของ 17 ปัจจัยมหภาค (%) ---
fig, ax = plt.subplots(figsize=(10, 7.2))
extreme_data_macro = []
for m in macro_names:
    if m in df_returns:
        m_ret = df_returns[m].dropna() * 100.0
        if len(m_ret) > 1:
            extreme_data_macro.append({'Macro': m, 'Max_Gain': m_ret.max(), 'Max_Loss': m_ret.min()})

df_ext_macro = pd.DataFrame(extreme_data_macro).set_index('Macro').sort_values('Max_Gain', ascending=True)
y_pos = np.arange(len(df_ext_macro))

ax.barh(y_pos, df_ext_macro['Max_Gain'], color='#388e3c', edgecolor='black', lw=0.5, height=0.68, label='วันที่กระชากขึ้นแรงสุด (Best Day %)')
ax.barh(y_pos, df_ext_macro['Max_Loss'], color='#d32f2f', edgecolor='black', lw=0.5, height=0.68, label='วันที่ร่วงดิ่งลึกสุด (Worst Day %)')

for i, (gain, loss) in enumerate(zip(df_ext_macro['Max_Gain'], df_ext_macro['Max_Loss'])):
    ax.text(gain + 1.0, i, f"+{gain:.1f}%", va='center', fontsize=9, fontweight='bold', color='#1b5e20')
    ax.text(loss - 1.0, i, f"{loss:.1f}%", va='center', ha='right', fontsize=9, fontweight='bold', color='#b71c1c')

ax.set_yticks(y_pos)
ax.set_yticklabels(df_ext_macro.index, fontproperties=thai_font_bold, fontsize=10.5)
ax.axvline(0, color='black', lw=1)
ax.set_title('วันที่ผันผวนสุดขีด: 17 ปัจจัยมหภาค (%)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xlabel('การเปลี่ยนแปลงในวันเดียว (%)', fontproperties=thai_font, fontsize=11)
ax.legend(loc='lower left', prop=thai_font, fontsize=9.5)
ax.set_xlim(-60, 125)
plt.tight_layout()

plt.savefig(output_fig_dir / "macro_04_extreme_days.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: macro_04_extreme_days.png")
plt.show()


In [ ]:
# --- 3.5 สหสัมพันธ์ระหว่าง 17 ปัจจัยมหภาคด้วยกันเอง ---
fig, ax = plt.subplots(figsize=(9, 7.5))
corr_macro = df_returns[macro_names].corr()
sns.heatmap(corr_macro, annot=False, cmap='coolwarm', ax=ax, cbar=True, center=0)
ax.set_title('สหสัมพันธ์ระหว่าง 17 ปัจจัยมหภาคด้วยกันเอง (Macro Correlation Matrix)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_xticklabels(ax.get_xticklabels(), fontproperties=thai_font_bold, rotation=45)
ax.set_yticklabels(ax.get_yticklabels(), fontproperties=thai_font_bold, rotation=0)
plt.tight_layout()

plt.savefig(output_fig_dir / "macro_05_correlation_matrix.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: macro_05_correlation_matrix.png")
plt.show()


---
# 🎯 ส่วนที่ 4: สไลด์วิเคราะห์เชิงลึก (Slide 2 - 5 แบบรูปเดี่ยว)


In [ ]:
# --- Slide 2: สหสัมพันธ์ 17 ปัจจัยมหภาค vs พอร์ตนำเข้า และ ส่งออก ---
returns_df = df_returns.copy()
returns_df['Import_Avg'] = returns_df[[s for s in import_names if s in returns_df.columns]].mean(axis=1)
returns_df['Export_Avg'] = returns_df[[s for s in export_names if s in returns_df.columns]].mean(axis=1)

corr_portfolio = pd.DataFrame(index=macro_names, columns=['Import_Avg', 'Export_Avg'])
for m in macro_names:
    if m in returns_df.columns:
        corr_portfolio.loc[m, 'Import_Avg'] = returns_df['Import_Avg'].corr(returns_df[m])
        corr_portfolio.loc[m, 'Export_Avg'] = returns_df['Export_Avg'].corr(returns_df[m])

corr_portfolio = corr_portfolio.astype(float)
corr_sorted = corr_portfolio.sort_values(by='Import_Avg', ascending=True)

fig, ax = plt.subplots(figsize=(10.5, 7.5))
y_pos = np.arange(len(corr_sorted))
bar_height = 0.38

bars_import = ax.barh(y_pos + bar_height/2, corr_sorted['Import_Avg'], height=bar_height, 
                       color='#1f77b4', label='พอร์ตหุ้นนำเข้าเฉลี่ย (Import)', alpha=0.88, edgecolor='black', lw=0.5)
bars_export = ax.barh(y_pos - bar_height/2, corr_sorted['Export_Avg'], height=bar_height, 
                       color='#2ca02c', label='พอร์ตหุ้นส่งออกเฉลี่ย (Export)', alpha=0.88, edgecolor='black', lw=0.5)

for bar in bars_import:
    w = bar.get_width()
    ax.text(w + (0.015 if w >= 0 else -0.045), bar.get_y() + bar.get_height()/2, 
            f"{w:.2f}", va='center', fontsize=9.5, fontweight='bold', color='#0d47a1')

for bar in bars_export:
    w = bar.get_width()
    ax.text(w + (0.015 if w >= 0 else -0.045), bar.get_y() + bar.get_height()/2, 
            f"{w:.2f}", va='center', fontsize=9.5, fontweight='bold', color='#1b5e20')

ax.set_yticks(y_pos)
ax.set_yticklabels(corr_sorted.index, fontproperties=thai_font_bold, fontsize=11)
ax.axvline(0, color='black', lw=1.2)
ax.set_xlabel('ค่าสัมประสิทธิ์สหสัมพันธ์ (Correlation Coefficient)', fontproperties=thai_font, fontsize=12)
ax.set_title('เปรียบเทียบความสัมพันธ์ของ 17 ปัจจัยมหภาค ระหว่างพอร์ตนำเข้า vs พอร์ตส่งออก', 
             fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.legend(loc='lower right', prop=thai_font, framealpha=0.95)
ax.set_xlim(-0.45, 0.78)
plt.tight_layout()

plt.savefig(output_fig_dir / "slide2_macro_portfolio_comparison.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: slide2_macro_portfolio_comparison.png")
plt.show()


In [ ]:
# --- Slide 3A: ดัชนีชิปโลก (SOX) vs หุ้นอิเล็กทรอนิกส์ ---
fig, ax = plt.subplots(figsize=(8.5, 5))
tech_stocks = ['HANA', 'KCE', 'DELTA', 'COM7', 'CPF', 'TU', 'BDMS']
sox_corr = [returns_df[s].corr(returns_df['SOX']) for s in tech_stocks]
colors_a = ['#e65100' if s in ['HANA', 'KCE', 'DELTA'] else '#90a4ae' for s in tech_stocks]

bars = ax.bar(tech_stocks, sox_corr, color=colors_a, edgecolor='black', lw=0.6, alpha=0.88)
ax.set_title('SOX Index vs หุ้นไทย (กลุ่มอิเล็กทรอนิกส์ตอบสนองสูงสุด)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_ylabel('Correlation กับ SOX', fontproperties=thai_font, fontsize=11)
ax.axhline(0.3, color='red', linestyle='--', alpha=0.7, label='Significant (+0.30)')
ax.set_ylim(-0.05, 0.52)

for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h + 0.015, f"{h:.2f}", ha='center', fontsize=10, fontweight='bold')

ax.legend(loc='upper right', prop=thai_font)
plt.tight_layout()

plt.savefig(output_fig_dir / "slide3a_sox_electronics.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: slide3a_sox_electronics.png")
plt.show()


In [ ]:
# --- Slide 3B: ราคาน้ำมันดิบ WTI vs หุ้นโรงกลั่น/ปิโตรเคมี ---
fig, ax = plt.subplots(figsize=(8.5, 5))
oil_stocks = ['PTTGC', 'TOP', 'IVL', 'IRPC', 'BGRIM', 'EGCO', 'TU']
wti_corr = [returns_df[s].corr(returns_df['WTI']) for s in oil_stocks]
colors_b = ['#1565c0' if s in ['PTTGC', 'TOP', 'IVL', 'IRPC'] else '#b0bec5' for s in oil_stocks]

bars = ax.bar(oil_stocks, wti_corr, color=colors_b, edgecolor='black', lw=0.6, alpha=0.88)
ax.set_title('ราคาน้ำมัน WTI vs หุ้นไทย (ผลบวกเฉพาะกลุ่มปิโตร/โรงกลั่น)', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_ylabel('Correlation กับ WTI', fontproperties=thai_font, fontsize=11)
ax.axhline(0.3, color='red', linestyle='--', alpha=0.7, label='Significant (+0.30)')
ax.set_ylim(-0.05, 0.55)

for b in bars:
    h = b.get_height()
    ax.text(b.get_x() + b.get_width()/2, h + 0.015, f"{h:.2f}", ha='center', fontsize=10, fontweight='bold')

ax.legend(loc='upper right', prop=thai_font)
plt.tight_layout()

plt.savefig(output_fig_dir / "slide3b_wti_energy.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: slide3b_wti_energy.png")
plt.show()


In [ ]:
# --- Slide 3C: อิทธิพลของค่าเงิน (DXY & CNY) vs หุ้นส่งออกเด่น ---
fig, ax = plt.subplots(figsize=(7, 5))
exp_focus = ['ITC', 'AAI', 'KCE', 'HANA', 'TU', 'STA']
fx_data = {
    'DXY (US Dollar)': [returns_df[s].corr(returns_df['DXY']) for s in exp_focus],
    'CNY/THB (หยวนจีน)': [returns_df[s].corr(returns_df['CNY_THB']) for s in exp_focus]
}
df_fx_corr = pd.DataFrame(fx_data, index=exp_focus)
sns.heatmap(df_fx_corr, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax, cbar=True,
            annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('อิทธิพลของค่าเงิน (DXY & CNY) ต่อหุ้นกลุ่มส่งออกเด่น', fontproperties=thai_font_bold, fontsize=13, pad=12)
ax.set_yticklabels(ax.get_yticklabels(), fontproperties=thai_font_bold, rotation=0)
plt.tight_layout()

plt.savefig(output_fig_dir / "slide3c_currency_exporters.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: slide3c_currency_exporters.png")
plt.show()


In [ ]:
# --- Slide 4A: ดอกเบี้ยนโยบาย TH_RATE สหสัมพันธ์ใกล้ศูนย์ (Time-Lag) ---
fig, ax = plt.subplots(figsize=(8.5, 6))
stocks_all = [s for s in import_names + export_names if s in returns_df.columns]
th_rate_corr = [returns_df[s].corr(returns_df['TH_RATE']) for s in stocks_all]

ax.barh(stocks_all, th_rate_corr, color='#78909c', edgecolor='black', lw=0.5, alpha=0.85)
ax.axvline(0, color='red', linestyle='-', lw=1.2)
ax.axvspan(-0.05, 0.05, color='yellow', alpha=0.2, label='Neutral Zone (-0.05 ถึง +0.05)')
ax.set_title('สหสัมพันธ์ของอัตราดอกเบี้ยนโยบาย (TH_RATE) กองอยู่ใกล้ศูนย์\nสะท้อนว่านโยบายการเงินมี Time-Lag Effect', 
             fontproperties=thai_font_bold, fontsize=12.5, pad=10)
ax.set_xlabel('Correlation กับ TH_RATE (รายวัน)', fontproperties=thai_font, fontsize=11)
ax.set_xlim(-0.15, 0.15)
ax.legend(loc='lower right', prop=thai_font)
plt.tight_layout()

plt.savefig(output_fig_dir / "slide4a_th_rate_lag.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: slide4a_th_rate_lag.png")
plt.show()


In [ ]:
# --- Slide 4B: DELTA Outlier (Market Beta เทียบกับ EGCO) ---
fig, ax = plt.subplots(figsize=(8.5, 6))
df_plot = returns_df[['SET50', 'DELTA', 'EGCO']].dropna()
corr_delta = df_plot['DELTA'].corr(df_plot['SET50'])
corr_egco = df_plot['EGCO'].corr(df_plot['SET50'])

sns.regplot(data=df_plot, x='SET50', y='EGCO', ax=ax, color='#1f77b4', 
            scatter_kws={'alpha': 0.2, 's': 10}, line_kws={'label': f'EGCO (นำเข้า, Corr = {corr_egco:.2f})', 'lw': 2})
sns.regplot(data=df_plot, x='SET50', y='DELTA', ax=ax, color='#e53935', 
            scatter_kws={'alpha': 0.2, 's': 10}, line_kws={'label': f'DELTA (ส่งออก, Corr = {corr_delta:.2f})', 'lw': 2})

ax.set_title('เปรียบเทียบ Market Beta: EGCO เกาะตลาด (+0.68) vs DELTA มีพฤติกรรมเฉพาะตัว (+0.08)', 
             fontproperties=thai_font_bold, fontsize=12.5, pad=10)
ax.set_xlabel('ผลตอบแทนดัชนี SET50', fontproperties=thai_font, fontsize=11)
ax.set_ylabel('ผลตอบแทนรายหุ้น', fontproperties=thai_font, fontsize=11)
ax.legend(loc='upper left', prop=thai_font, facecolor='white', framealpha=0.95)
ax.set_xlim(-0.06, 0.06)
ax.set_ylim(-0.15, 0.15)
plt.tight_layout()

plt.savefig(output_fig_dir / "slide4b_delta_market_beta.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: slide4b_delta_market_beta.png")
plt.show()


In [ ]:
# --- Slide 5: สะพานเชื่อมจาก EDA สู่ Feature Engineering ---
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(15, 6.5))
ax.axis('off')

boxes = [
    {"xy": (0.05, 0.70), "w": 0.26, "h": 0.22, "color": "#ffe0b2", "edge": "#f57c00",
     "title": "🔍 ข้อค้นพบจาก EDA (Findings)", "body": "1. อัตราดอกเบี้ยและ Yield มี\n   Correlation รายวันต่ำมาก\n2. นโยบายการเงินมี Time-Lag\n   ไม่ส่งผลกระทบทันทีในวันเดียว"},
    {"xy": (0.05, 0.38), "w": 0.26, "h": 0.22, "color": "#e1f5fe", "edge": "#0288d1",
     "title": "🔍 ข้อค้นพบจาก EDA (Findings)", "body": "1. SOX มีผลสูงเฉพาะอิเล็กทรอนิกส์\n2. WTI มีผลสูงเฉพาะปิโตร/พลังงาน\n3. DXY หนุนกลุ่มส่งออกอาหารสัตว์"},
    {"xy": (0.05, 0.06), "w": 0.26, "h": 0.22, "color": "#ffcdd2", "edge": "#d32f2f",
     "title": "🔍 ข้อค้นพบจาก EDA (Findings)", "body": "1. VIX ส่งผลลบต่อหุ้นไทยทั้งกระดาน\n2. DELTA มีความผันผวนเฉพาะตัวสูง\n   ไม่ได้อิงตามตลาด SET50 เสมอไป"},

    {"xy": (0.38, 0.70), "w": 0.28, "h": 0.22, "color": "#fff3e0", "edge": "#f57c00",
     "title": "⚙️ การสร้างฟีเจอร์ (Feature Action)", "body": "• สร้าง Time-Lag 1, 3, 5, 10 วัน\n• คำนวณ Yield Curve Slope\n  (TH_10Y - TH_20Y, US_10Y - US_30Y)\n• Rolling Moving Average (5, 20 วัน)"},
    {"xy": (0.38, 0.38), "w": 0.28, "h": 0.22, "color": "#e0f7fa", "edge": "#00acc1",
     "title": "⚙️ การสร้างฟีเจอร์ (Feature Action)", "body": "• จัดกลุ่ม Domain-Specific Features\n• คำนวณ Spread WTI vs Brent\n• FX Momentum & Relative Strength\n  (THB vs DXY, CNY, JPY)"},
    {"xy": (0.38, 0.06), "w": 0.28, "h": 0.22, "color": "#ffebee", "edge": "#e53935",
     "title": "⚙️ การสร้างฟีเจอร์ (Feature Action)", "body": "• Rolling Volatility (5, 20 วัน)\n• Technical Momentum: RSI (14 วัน)\n• สัญญาณ Risk-off Trigger จาก VIX\n  เพื่อตรวจจับสภาวะวิกฤตตลาด"},

    {"xy": (0.73, 0.70), "w": 0.23, "h": 0.22, "color": "#e8f5e9", "edge": "#388e3c",
     "title": "🎯 ประโยชน์ต่อโมเดล (Impact)", "body": "ดักจับผลกระทบย้อนหลัง\nของนโยบายดอกเบี้ย\nและเพิ่มพลังการพยากรณ์"},
    {"xy": (0.73, 0.38), "w": 0.23, "h": 0.22, "color": "#e8f5e9", "edge": "#388e3c",
     "title": "🎯 ประโยชน์ต่อโมเดล (Impact)", "body": "ลดสัญญาณรบกวน (Noise)\nโดยคัดเลือกฟีเจอร์ที่ตรงกับ\nโครงสร้างธุรกิจของหุ้นนั้นๆ"},
    {"xy": (0.73, 0.06), "w": 0.23, "h": 0.22, "color": "#e8f5e9", "edge": "#388e3c",
     "title": "🎯 ประโยชน์ต่อโมเดล (Impact)", "body": "ช่วยให้ Classifier ตัดสินใจได้\nแม่นยำขึ้นเมื่อเกิดสภาวะตลาด\nผันผวนหรือตื่นตระหนก"}
]

for b in boxes:
    rect = patches.FancyBboxPatch(b["xy"], b["w"], b["h"], boxstyle="round,pad=0.02",
                                  fc=b["color"], ec=b["edge"], lw=1.5)
    ax.add_patch(rect)
    ax.text(b["xy"][0] + 0.015, b["xy"][1] + b["h"] - 0.045, b["title"], 
            fontproperties=thai_font_bold, fontsize=11, color='#1a237e')
    ax.text(b["xy"][0] + 0.015, b["xy"][1] + 0.03, b["body"], 
            fontproperties=thai_font, fontsize=9.5, linespacing=1.45, va='bottom')

for y in [0.81, 0.49, 0.17]:
    ax.annotate('', xy=(0.37, y), xytext=(0.32, y),
                arrowprops=dict(facecolor='#546e7a', edgecolor='none', width=2.5, headwidth=8))
    ax.annotate('', xy=(0.72, y), xytext=(0.67, y),
                arrowprops=dict(facecolor='#546e7a', edgecolor='none', width=2.5, headwidth=8))

ax.set_title("บทสรุปที่ส่งต่อไปยัง Feature Engineering (Bridge from EDA to ML Pipeline)", 
             fontproperties=thai_font_bold, fontsize=15, pad=10)
plt.tight_layout()

plt.savefig(output_fig_dir / "slide5_eda_to_features_bridge.png", dpi=300, bbox_inches='tight')
print("✓ บันทึกภาพ: slide5_eda_to_features_bridge.png")
plt.show()


---
## 🏁 สรุปรายการภาพเดี่ยวทั้งหมด (21 ภาพเดี่ยวพร้อมวางบนสไลด์ทันที):

### 🏢 1. หุ้นกลุ่มนำเข้า (Import Stocks)
1. `import_01_price_trend.png` — กราฟเส้นแนวโน้มราคา 10 ปี (Base 100)
2. `import_02_total_return.png` — กราฟแท่งผลตอบแทนสะสมตลอดช่วงเวลา (%)
3. `import_03_up_down_ratio.png` — กราฟแท่งสัดส่วนวันปิดบวก vs ปิดลบ (%)
4. `import_04_extreme_days.png` — กราฟแท่งวันที่ขึ้นแรงสุด vs ลงลึกสุดใน 1 วัน (%)
5. `import_05_correlation_matrix.png` — Heatmap สหสัมพันธ์ภายในกลุ่มนำเข้า

### 🚢 2. หุ้นกลุ่มส่งออก (Export Stocks)
6. `export_01_price_trend.png` — กราฟเส้นแนวโน้มราคา 10 ปี (Base 100)
7. `export_02_total_return.png` — กราฟแท่งผลตอบแทนสะสมตลอดช่วงเวลา (%)
8. `export_03_up_down_ratio.png` — กราฟแท่งสัดส่วนวันปิดบวก vs ปิดลบ (%)
9. `export_04_extreme_days.png` — กราฟแท่งวันที่ขึ้นแรงสุด vs ลงลึกสุดใน 1 วัน (%)
10. `export_05_correlation_matrix.png` — Heatmap สหสัมพันธ์ภายในกลุ่มส่งออก

### 🌐 3. ปัจจัยมหภาค (17 Macro Factors)
11. `macro_01_price_trend.png` — กราฟเส้นแนวโน้มดัชนีสำคัญ (Base 100)
12. `macro_02_total_return.png` — กราฟแท่งการเปลี่ยนแปลงสะสม (%)
13. `macro_03_up_down_ratio.png` — กราฟแท่งสัดส่วนวันขึ้น vs วันลง (%)
14. `macro_04_extreme_days.png` — กราฟแท่งวันที่ผันผวนสุดขีด (%)
15. `macro_05_correlation_matrix.png` — Heatmap สหสัมพันธ์ระหว่าง 17 ปัจจัยมหภาค

### 🎯 4. สไลด์เจาะลึกและโมเดล (Slide 2-5)
16. `slide2_macro_portfolio_comparison.png` — เปรียบเทียบ 17 ปัจจัยมหภาคกับพอร์ตนำเข้า vs ส่งออก
17. `slide3a_sox_electronics.png` — ดัชนีชิปโลก SOX vs หุ้นอิเล็กทรอนิกส์
18. `slide3b_wti_energy.png` — ราคาน้ำมัน WTI vs หุ้นโรงกลั่น/ปิโตรเคมี
19. `slide3c_currency_exporters.png` — ค่าเงิน DXY & CNY vs หุ้นส่งออก
20. `slide4a_th_rate_lag.png` — ดอกเบี้ยนโยบาย TH_RATE สหสัมพันธ์ใกล้ศูนย์ (Time-Lag)
21. `slide4b_delta_market_beta.png` — Market Beta: DELTA vs EGCO เทียบกับ SET50
22. `slide5_eda_to_features_bridge.png` — แผนผังสรุปการเชื่อมโยงสู่ Feature Engineering
